In [18]:
import random
from itertools import product
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from env_raw import QECMaintenanceEnv
import torch.optim as optim
from joblib import Parallel, delayed

# Selecting the GPU for the device if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu



# Setting seeds for reproducibility 


In [ ]:
# Reproducability 
def set_seed(seed):
    # Seed python, numpy, and torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# The functions needed for the observations and history of the environment 

In [ ]:
# Converting the observation to a list of the Z-ancilla values
def obs_to_bit_flips(obs, bit_flip_indices, num_ancilla):
    # All stabilizer observations until our current time step 
    syndrome_history = obs[:-num_ancilla]
    # Reshape according to the timestep in which the syndrome was observed
    rounds = syndrome_history.reshape(-1, num_ancilla)
    # Consider only the last time step's syndromes
    latest_round = rounds[-1]
    # Consider only the relevant syndromes
    z_ancillas = latest_round[bit_flip_indices]
    return z_ancillas.astype(np.uint8)

def initialize_obs_history(obs, history_len):
    # Initialize with repeated first observation
    return [obs.copy() for _ in range(history_len)]

def update_obs_history(history, obs):
    # Drop oldest, append newest
    return history[1:] + [obs.copy()]

def history_to_input(history):
    # Concatenate along feature dimension
    return np.concatenate(history, axis=0)

# Convert the observation into a history index
def initialize_history_state(obs, bit_flip_indices, num_ancilla, history_len):
    # Extract the binary syndrome pattern
    bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, num_ancilla)
    # Initialize history with copies of the vector
    history = [bit_flip_measures.copy() for _ in range(history_len)]
    return history

def update_history_state(obs, history, bit_flip_indices, num_ancilla):
    # Extract the current binary syndrome pattern
    bit_flip_measures = obs_to_bit_flips(obs, bit_flip_indices, num_ancilla)
    # Update history by dropping oldest and appending new vector
    history = history[1:] + [bit_flip_measures.copy()]
    return history


# PPO Network 

In [21]:
class PPONetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=512):
        super().__init__()
        # Shared layers of the actor-critic network
        # Different from cleanrl because of observation space size difference
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        # Add a seperate final layer for both networks
        # Actor output is the action logits
        self.actor = nn.Linear(hidden_dim, n_actions)
        # Critic outputs the state value
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        # Map the state index to a learned vector
        x = obs.float()
        # Process the embedding with the shared network
        h = self.shared(x)
        # Action logits and state value
        logits = self.actor(h)
        value = self.critic(h).squeeze(-1)
        return logits, value


# Evaluation model

In [22]:
# Evaluation episodes are different from the training episodes
@torch.inference_mode()
def evaluate_ppo(
    env,
    agent,
    bit_flip_indices,
    history_len,
    num_basic_states,
    # We evaluate over 30 episodes with the current knowledge to average over them
    num_episodes=30,
):
    episode_returns = []

    for _ in range(num_episodes):
        # We reset the environment so we do not start with errors
        obs, _ = env.reset()
        done = truncated = False

        history = initialize_obs_history(obs, history_len)

        episode_reward = 0

        while not (done or truncated):
            # Greedy evaluation by taking the highest logit action
            state_input = history_to_input(history)
            state_tensor = torch.tensor(state_input, dtype=torch.float32, device=device).unsqueeze(0)
            logits, _ = agent(state_tensor)
            action = torch.argmax(logits, dim=-1).item()

            # Step the environment with the chosen action
            obs, _, done, truncated, _ = env.step(action)
            # In evaluation we count how long the agent survives, not the environment reward
            episode_reward += 1

            if not (done or truncated):
                # Update the history only if the episode continues
                history = update_obs_history(history, obs)

        episode_returns.append(episode_reward)

    return float(np.mean(episode_returns))


# Training Model

In [ ]:
def train_ppo(
# Values taken from cleanrl
env,
total_timesteps=200_000,
rollout_size=2048,
minibatch_size=512,
update_epochs=10,
learning_rate=2.5e-4,
gamma=0.99,
gae_lambda=0.95,
clip_coef=0.1,
entropy_coef=0.01,
value_coef=0.5,
max_grad_norm=0.5,
history_len=1,
eval_every_steps=1000,
eval_episodes=30,
hidden_dim=512,
use_compile=False,
):
    # We consider only bit flips
    bit_flip_indices = [i for i, t in enumerate(env.lattice["ancilla_types"]) if t == "Z"]
    # Each stablilizer can either show a syndrome or not
    num_basic_states = 2 ** len(bit_flip_indices)
    # Initialize the first environment state before collecting rollouts
    obs, _ = env.reset()
    # We use only the Z-ancilla observations and consider history length
    state_dim = len(obs) * history_len
    # Flip any physical qubit, do nothing, or truncate 
    n_actions = env.num_data + 2
    # Initialize the neural network
    agent = PPONetwork(
        state_dim=state_dim,
        n_actions=n_actions,
        hidden_dim=hidden_dim,
        ).to(device)

    if False and hasattr(torch, "compile"):
        try:
            # Compile the model when available for faster execution
            agent = torch.compile(agent)
        except Exception:
            pass

    optimizer = optim.Adam(agent.parameters(), lr=learning_rate)

    # Store training and evaluation statistics
    training_episode_rewards = []
    evaluation_returns = []
    losses = []

    history = initialize_obs_history(obs, history_len)

    current_episode_reward = 0.0
    episode_count = 0
    num_steps = 0

    last_eval_step = 0
    while num_steps < total_timesteps:
        # Shorten the final rollout if we are near the total timestep budget
        current_rollout_size = min(rollout_size, total_timesteps - num_steps)
        # Tensors to store the rollout
        rollout_states = torch.empty((current_rollout_size, state_dim), dtype=torch.float32, device=device)
        rollout_actions = torch.empty(current_rollout_size, dtype=torch.long, device=device)
        rollout_logprobs = torch.empty(current_rollout_size, dtype=torch.float32, device=device)
        rollout_rewards = torch.empty(current_rollout_size, dtype=torch.float32, device=device)
        rollout_dones = torch.empty(current_rollout_size, dtype=torch.float32, device=device)
        rollout_values = torch.empty(current_rollout_size, dtype=torch.float32, device=device)

        for t in range(current_rollout_size):
            # We want to consider the entire state as input to allow the NN to generalize
            state_input = history_to_input(history)
            state_tensor = torch.tensor(state_input, dtype=torch.float32, device=device).unsqueeze(0)

            with torch.inference_mode():
                # Sample from the current policy and record its log probability
                logits, value = agent(state_tensor)
                dist = torch.distributions.Categorical(logits=logits)
                action = dist.sample()
                logprob = dist.log_prob(action)
            # Make a step with the environment 
            next_obs, reward, terminated, truncated, _ = env.step(action.item())
            # Update history
            history = update_obs_history(history, next_obs)
            # Check if we continue
            done = terminated or truncated

            # Save the new transition in the rollout buffer
            rollout_states[t] = state_tensor[0]
            rollout_actions[t] = action[0]
            rollout_logprobs[t] = logprob[0]
            rollout_rewards[t] = reward
            rollout_dones[t] = float(done)
            rollout_values[t] = value[0]

            current_episode_reward += reward
            num_steps += 1

            if done:
                # Store the full episode reward once the episode ends
                training_episode_rewards.append(current_episode_reward)
                episode_count += 1

                if num_steps - last_eval_step >= eval_every_steps:
                    # Evaluate the current performance of the agent
                    eval_return = evaluate_ppo(
                        env=env,
                        agent=agent,
                        bit_flip_indices=bit_flip_indices,
                        history_len=history_len,
                        num_basic_states=num_basic_states,
                        num_episodes=eval_episodes,
                    )
                    evaluation_returns.append(eval_return)
                    # Print progress
                    print(
                        f"episode: {episode_count} | "
                        f"evaluation return: {round(eval_return,2)} | "
                        f"last training reward: {round(current_episode_reward,2)} | "
                        f"steps: {num_steps}"
                    )
                    last_eval_step = num_steps

                # Reset the environment and rebuild the initial history
                current_episode_reward = 0.0
                obs, _ = env.reset()
                history = initialize_history_state(
                    obs=obs,
                    bit_flip_indices=bit_flip_indices,
                    num_ancilla=env.num_ancilla,
                    history_len=history_len,
                )
                history = initialize_obs_history(obs, history_len)
            else:
                # Continue the same episode and update the state history
                obs = next_obs

        with torch.inference_mode():
            # Bootstrap the value function from the last state of the rollout
            bootstrap_input = history_to_input(history)
            bootstrap_state = torch.tensor(bootstrap_input, dtype=torch.float32, device=device).unsqueeze(0)
            _, next_value = agent(bootstrap_state)
            next_value = next_value[0]
        # Compute generalized advantage estimates backwards through the rollout
        advantages = torch.empty(current_rollout_size, dtype=torch.float32, device=device)
        gae = torch.tensor(0.0, dtype=torch.float32, device=device)

        for t in reversed(range(current_rollout_size)):
            if t == current_rollout_size - 1:
                # For the last transition we use the bootstrap value
                next_non_terminal = 1.0 - rollout_dones[t]
                next_val = next_value
            else:
                # Otherwise use the value prediction at the next timestep
                next_non_terminal = 1.0 - rollout_dones[t]
                next_val = rollout_values[t + 1]

            delta = rollout_rewards[t] + gamma * next_val * next_non_terminal - rollout_values[t]
            gae = delta + gamma * gae_lambda * next_non_terminal * gae
            advantages[t] = gae

        # Returns are value targets for the critic
        returns = advantages + rollout_values
        # Normalize advantages for more stable policy updates
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Shuffle rollout indices before minibatch updates
        indices = torch.randperm(current_rollout_size, device=device)
        last_loss = None

        for _ in range(update_epochs):
            # Reshuffle before each epoch
            indices = indices[torch.randperm(current_rollout_size, device=device)]

            for start in range(0, current_rollout_size, minibatch_size):
                end = min(start + minibatch_size, current_rollout_size)
                mb_inds = indices[start:end]
                # Collect the current minibatch
                mb_states = rollout_states[mb_inds]
                mb_actions = rollout_actions[mb_inds]
                mb_old_logprobs = rollout_logprobs[mb_inds]
                mb_returns = returns[mb_inds]
                mb_advantages = advantages[mb_inds]

                # Run the current policy on the minibatch
                logits, values = agent(mb_states)
                dist = torch.distributions.Categorical(logits=logits)

                new_logprobs = dist.log_prob(mb_actions)
                entropy = dist.entropy().mean()

                # Clipped ppo policy objective
                ratio = torch.exp(new_logprobs - mb_old_logprobs)
                unclipped = ratio * mb_advantages
                clipped = torch.clamp(ratio, 1.0 - clip_coef, 1.0 + clip_coef) * mb_advantages
                policy_loss = -torch.min(unclipped, clipped).mean()

                # Train the critic to match the computed returns
                value_loss = 0.5 * ((values - mb_returns) ** 2).mean()
                # Combine policy, value, and entropy terms into loss
                loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                # Clip gradients 
                nn.utils.clip_grad_norm_(agent.parameters(), max_grad_norm)
                optimizer.step()

                last_loss = loss

        if last_loss is not None:
            losses.append(last_loss.item())

    return agent, evaluation_returns, training_episode_rewards, losses


# Plotting Functions

In [ ]:
def average_runs(run_list):
    if len(run_list) == 0:
        return np.array([])

    max_len = max(len(run) for run in run_list)
    arr = np.full((len(run_list), max_len), np.nan, dtype=np.float32)

    for i, run in enumerate(run_list):
        arr[i, :len(run)] = run

    return np.nanmean(arr, axis=0)


def compare_history_0v1(
    total_timesteps=40_000,
    eval_every_steps=1000,
    eval_episodes=30,
    seed=0,
):
    results = {}

    for mode in ["history", "no_history"]:

        set_seed(seed)
        env = QECMaintenanceEnv(render_mode=None)

        if mode == "no_history":
            history_len = 1 
        else:
            history_len = 2 

        agent, eval_returns, train_rewards, losses = train_ppo(
            env=env,
            total_timesteps=total_timesteps,
            eval_every_steps=eval_every_steps,
            eval_episodes=eval_episodes,
            history_len=history_len,
            learning_rate=3e-4,
            entropy_coef=0.005,
        )

        env.close()

        # Save raw data for later
        np.save(f"{mode}_eval.npy", np.array(eval_returns))
        np.save(f"{mode}_train.npy", np.array(train_rewards))


        results[mode] = {
            "eval": eval_returns,
            "train": train_rewards,
            "loss": losses,
        }

    return results


# Run across the seeds in parallel for history vs no history

In [ ]:
# Zero and 1 history observation
def run_single_seed(seed, total_timesteps, eval_every_steps, eval_episodes):
    seed = int(seed)
    set_seed(seed)

    results = compare_history_0v1(
        total_timesteps=total_timesteps,
        eval_every_steps=eval_every_steps,
        eval_episodes=eval_episodes,
        seed=seed,
    )

    return {
        "history": results["history"]["eval"],
        "no_history": results["no_history"]["eval"],
    }

def ppo_parallel(
    seeds,
    total_timesteps=1_000_001,
    eval_every_steps=300,
    eval_episodes=30,
    n_jobs=15,
):
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_single_seed)(
            seed,
            total_timesteps,
            eval_every_steps,
            eval_episodes,
        )
        for seed in seeds
    )

    history_all = [r["history"] for r in results]
    no_history_all = [r["no_history"] for r in results]

    return {
        "history_mean": average_runs(history_all),
        "no_history_mean": average_runs(no_history_all),
        "raw": results,
    }  
results = ppo_parallel(seeds=np.random.randint(100, size=15))

np.save("ppo.npy", results, allow_pickle=True)

# Edited to include history of two and three time-steps

In [ ]:
# 2 and 3 previous observations
def compare_history_2v3(
    total_timesteps=40_000,
    eval_every_steps=1000,
    eval_episodes=30,
    seed=0,
):
    results = {}
    for mode in ["history_2", "history_3"]:
        set_seed(seed)
        env = QECMaintenanceEnv(render_mode=None)

        if mode == "history_2":
            history_len = 3
        else:
            history_len = 4

        agent, eval_returns, train_rewards, losses = train_ppo(
            env=env,
            total_timesteps=total_timesteps,
            eval_every_steps=eval_every_steps,
            eval_episodes=eval_episodes,
            history_len=history_len,
            learning_rate=3e-4,
            entropy_coef=0.005,
        )

        env.close()

        # Save raw data for later
        np.save(f"{mode}_eval.npy", np.array(eval_returns))
        np.save(f"{mode}_train.npy", np.array(train_rewards))

        results[mode] = {
            "eval": eval_returns,
            "train": train_rewards,
            "loss": losses,
        }


    return results


def plot_comparison(results, eval_every_steps=100):
    plt.figure(figsize=(8, 5))

    for mode, label in [("history_mean", "history"), ("no_history_mean", "no_history")]:
        curve = results[mode]
        x = np.arange(len(curve)) * eval_every_steps
        plt.plot(x, curve, label=label)

    plt.xlabel("Training episodes")
    plt.ylabel("Cumulative survival steps (mean over seeds)")
    plt.title("History vs No-History (Averaged over seeds)")
    plt.legend()
    plt.tight_layout()
    plt.show()

def run_single_seed(seed, total_timesteps, eval_every_steps, eval_episodes):
    seed = int(seed)
    set_seed(seed)

    results = compare_history_2v3(
        total_timesteps=total_timesteps,
        eval_every_steps=eval_every_steps,
        eval_episodes=eval_episodes,
        seed=seed,
    )

    return {
    "history_2": results["history_2"]["eval"],
    "history_3": results["history_3"]["eval"],
    }

def run_ppo_parallel(
    seeds,
    total_timesteps=1_000_001,
    eval_every_steps=5000,
    eval_episodes=30,
    n_jobs=15,
):
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_single_seed)(
            seed,
            total_timesteps,
            eval_every_steps,
            eval_episodes,
        )
        for seed in seeds
    )

    history_2_all = [r["history_2"] for r in results]
    history_3_all = [r["history_3"] for r in results]

    return {
    "history_2_mean": average_runs(history_2_all),
    "history_3_mean": average_runs(history_3_all),
    "raw": results,
    }
results = run_ppo_parallel(seeds=np.random.randint(100, size=15))

np.save("ppo_history.npy", results, allow_pickle=True)

# Plotting results

In [ ]:
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt
def smooth_curve(values, window=15, polyorder=1):
    return savgol_filter(values, window_length=window, polyorder=polyorder)


def log_and_plot(eval_every_steps=5000):
    
    results_long = np.load("ppo.npy", allow_pickle=True).item()
    
    smoothing_window = 35

    history_smoothed = smooth_curve(results_long["history_mean"], window=smoothing_window)
    no_history_smoothed = smooth_curve(results_long["no_history_mean"], window=smoothing_window)


    x_no_hist = np.arange(len(no_history_smoothed)) * 5000
    x_hist1 = np.arange(len(history_smoothed)) * 5000


    print("no history:", round(no_history_smoothed[-1],2))
    print("history 1:", round(history_smoothed[-1],2))


    results = np.load("ppo_history.npy", allow_pickle=True).item()
    
    history_3 = results["history_2_mean"]
    history_4 = results["history_3_mean"]
    
    his_3_smoothed = smooth_curve(history_3, window=smoothing_window)
    his_4_smoothed = smooth_curve(history_4, window=smoothing_window)

    x = np.arange(len(history_3)) * eval_every_steps
    
    
    print("history of two time-steps:", round(his_3_smoothed[-1],2))
    print("history of three time-steps:", round(his_4_smoothed[-1],2))


    plt.figure(figsize=(8, 5))
    plt.ylim(bottom=10, top=100)
     
    plt.plot(x_no_hist, no_history_smoothed, label="no history")
    plt.plot(x_hist1, history_smoothed, label="history 1")

    plt.plot(x_no_hist, his_3_smoothed, label="history 2")
    plt.plot(x_no_hist, his_4_smoothed, label="history 3")
    
    
    
    plt.xlabel("Time steps (in millions)", fontsize=14)
    plt.ylabel("Survival length (in time-steps)", fontsize=14)
    plt.title("PPO performance with different history lengths", fontsize=16)
    plt.legend(prop = { "size": 14 })
    plt.tight_layout()
    plt.show()


log_and_plot()